In [ ]:
# If needed

# import os
# os.chdir(".../atmosphere-profile-retrieval-dense-nn")
# os.getcwd()

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt

import humanize as h
from tqdm.notebook import tqdm

from paths import LOG_DIR, CHECKPOINT_DIR, INPUTS_DIR, AH_INPUTS_TRAIN_PATH, BT_INPUTS_TRAIN_PATH, CZ_INPUTS_TRAIN_PATH, DATE_INPUTS_TRAIN_PATH, GHEIGHT_INPUTS_TRAIN_PATH, GAH_INPUTS_TRAIN_PATH, MM_INPUTS_TRAIN_PATH, AH_INPUTS_TEST_PATH, BT_INPUTS_TEST_PATH, CZ_INPUTS_TEST_PATH, DATE_INPUTS_TEST_PATH, GHEIGHT_INPUTS_TEST_PATH, GAH_INPUTS_TEST_PATH, MM_INPUTS_TEST_PATH
import training.models as models

seed = 42
torch.manual_seed(seed)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


def model_stats(model):
    total_params = sum(p.numel() for p in model.parameters())
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    
    print(f"Model size: {h.intword(total_params, format='%.2f')} parameters ({h.naturalsize(param_bytes, binary=True, format='%.2f')})\n")

In [ ]:
def generate_loaders(X_names, batch_size):
    if "AH" in X_names and "BT" in X_names:
        raise Exception("Model has to use either BT or AH as main input, but not both.")

    if "AH" not in X_names and "BT" not in X_names:
        raise Exception("Model has to use either BT or AH as main input, but not both.")
    
    name_paths_dict = {
        "BT": (BT_INPUTS_TRAIN_PATH, BT_INPUTS_TEST_PATH),
        "CZ": (CZ_INPUTS_TRAIN_PATH, CZ_INPUTS_TEST_PATH),
        "DATE": (DATE_INPUTS_TRAIN_PATH, DATE_INPUTS_TEST_PATH),
        "GH": (GHEIGHT_INPUTS_TRAIN_PATH, GHEIGHT_INPUTS_TEST_PATH),
        "GAH": (GAH_INPUTS_TRAIN_PATH, GAH_INPUTS_TEST_PATH),
        "MM": (MM_INPUTS_TRAIN_PATH, MM_INPUTS_TEST_PATH),
        "AH": (AH_INPUTS_TRAIN_PATH, AH_INPUTS_TEST_PATH)
    }
    
    X_list_train = []
    X_list_test = []
    for name in X_names:
        train_path, test_path = name_paths_dict[name]

        train_arr = np.load(train_path)
        test_arr = np.load(test_path)

        if name == "AH":
            train_arr = np.nan_to_num(train_arr, nan=0.0)
            test_arr = np.nan_to_num(test_arr, nan=0.0)

        X_list_train.append(train_arr)
        X_list_test.append(test_arr)
    
    X_train = np.hstack(X_list_train)
    X_test = np.hstack(X_list_test)

    if "BT" in X_names:
        Y_train = np.load(AH_INPUTS_TRAIN_PATH)
        Y_test = np.load(AH_INPUTS_TEST_PATH)
    else:
        Y_train = np.load(BT_INPUTS_TRAIN_PATH)
        Y_test = np.load(BT_INPUTS_TEST_PATH)

    train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    return train_loader, test_loader, X_train.shape[1], Y_train.shape[1]

In [ ]:
def masked_mse_loss(Y_pred, Y_true):
    mask = ~torch.isnan(Y_true)
    diff = Y_pred[mask] - Y_true[mask]
    
    return torch.mean(diff ** 2)



def train_loop(model, dataloader, loss_fn, optimizer):
    train_loss = 0
    
    size = len(dataloader.dataset)
    model.train()

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        cur_batch_size = X.size(0)
                
        pred = model(X)


        loss = loss_fn(pred, y)
        train_loss += loss.item() * cur_batch_size

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= size

    return train_loss


def eval_loop(model, dataloader, loss_fn):
    eval_loss = 0.

    size = len(dataloader.dataset)
    model.eval()

    with torch.no_grad():
        for X, Y in dataloader:
            X = X.to(device)
            Y = Y.to(device)
            
            cur_batch_size = X.size(0)
            
            pred = model(X)
            
            loss = loss_fn(pred, Y)
            eval_loss += loss.item() * cur_batch_size

        eval_loss /= size

    return eval_loss



def train_model(X_names, model, loss_fn, optim, train_loader, eval_loader, epochs):
    model_name = f"{'-'.join(X_names)}.{type(model).__name__}"
    log_dir = LOG_DIR / model_name
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}.pt"

    model = model.to(device)
    with SummaryWriter(log_dir=log_dir) as writer:
        eval_losses = []
        
        for epoch in tqdm(range(epochs), desc="Epoch"):
            train_loss = train_loop(model, train_loader, loss_fn, optim)
            eval_loss = eval_loop(model, eval_loader, loss_fn)
            eval_losses.append(eval_loss)
    
            writer.add_scalar("Train loss", train_loss, epoch+1)
            writer.add_scalar("Eval loss", eval_loss, epoch+1)
            writer.flush()

            if eval_loss == min(eval_losses):
                CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), checkpoint_path)

In [ ]:
def pipeline(X_names, model, train_loader, test_loader, epochs):
    loss = masked_mse_loss
    optim = torch.optim.Adam(model.parameters())
    
    model_stats(model)
    
    train_model(X_names, model, loss, optim, train_loader, test_loader, epochs)

In [ ]:
X_names = ["BT"]
batch_size = 32

train_loader, test_loader, features, targets = generate_loaders(X_names, batch_size)

In [ ]:
model = models.expanding_9(features, targets)
model_stats(model)

In [ ]:
epochs = 200

pipeline(X_names, model, train_loader, test_loader, epochs)